# 03 — Pre-DeFi Econometric Core Panel

**Purpose:** Build the hourly dataset used in ALL econometric analysis. Merges Bybit perp data with spot BTC/ETH benchmarks, computes all transformed variables, and saves the analysis-ready panel.

**Inputs:** master calendar + Bybit (klines, OI, funding) + spot (CCData BTC, ETH)
**Output:** `data/econ/econ_core_predefi_1h.parquet`

DeFi liquidation data will be merged in a later notebook (04). This notebook must run cleanly without it.

In [5]:
# ── Setup ──
import sys; sys.path.insert(0, "..")
from config import CFG, REPORTS_DIR, ECON_DIR
CFG.ensure_dirs()

import pandas as pd
import numpy as np
import json

print(f"Project root: {CFG.ROOT}")

Project root: .


In [9]:
# ── 1. Load all inputs ──

cal = pd.read_parquet(CFG.FILES.master_calendar, engine="pyarrow")
cal["date"] = pd.to_datetime(cal["date"], utc=True)

def load(path):
    df = pd.read_parquet(path, engine="pyarrow")
    df["date"] = pd.to_datetime(df["date"], utc=True)
    return df

bybit_k = load(CFG.FILES.bybit_klines)
bybit_oi = load(CFG.FILES.bybit_oi)
bybit_f = load(CFG.FILES.bybit_funding)
btc_spot = load(CFG.FILES.btc_spot)
eth_spot = load(CFG.FILES.eth_spot)

print("Inputs loaded:")
for name, df in [("calendar", cal), ("bybit_klines", bybit_k), ("bybit_oi", bybit_oi),
                  ("bybit_funding", bybit_f), ("btc_spot", btc_spot), ("eth_spot", eth_spot)]:
    print(f"  {name:16s}: {len(df):,} rows")

Inputs loaded:
  calendar        : 41,328 rows
  bybit_klines    : 41,328 rows
  bybit_oi        : 41,328 rows
  bybit_funding   : 41,328 rows
  btc_spot        : 41,328 rows
  eth_spot        : 41,328 rows


In [11]:
# ── 2. Standardize column names & merge ──

# Bybit klines → keep OHLCV
klines = bybit_k[["date", "close", "volume"]].rename(
    columns={"close": "close_perp", "volume": "volume_perp"}
)

# OI
oi_col = [c for c in bybit_oi.columns if c != "date"][0]
oi = bybit_oi[["date", oi_col]].rename(columns={oi_col: "oi"})

# Funding
f_col = [c for c in bybit_f.columns if c != "date"][0]
funding = bybit_f[["date", f_col]].rename(columns={f_col: "funding_rate"})

# Spot
btc = btc_spot[["date", "close"]].rename(columns={"close": "close_btc_spot"})
eth = eth_spot[["date", "close"]].rename(columns={"close": "close_eth_spot"})

# Merge everything onto calendar
panel = cal.copy()
for df in [klines, oi, funding, btc, eth]:
    panel = panel.merge(df, on="date", how="left")

print(f"Panel: {panel.shape[0]:,} rows × {panel.shape[1]} cols")

Panel: 41,328 rows × 7 cols


In [13]:
# ── 3. Variable transformations ──

# 3a. Log returns (×100 for readability in bps-like units)
panel["ret_eth_perp"]  = np.log(panel["close_perp"]).diff() * 100
panel["ret_btc_spot"]  = np.log(panel["close_btc_spot"]).diff() * 100
panel["ret_eth_spot"]  = np.log(panel["close_eth_spot"]).diff() * 100

# 3b. Open Interest: first difference + z-score + regime flag
from config import CFG, REPORTS_DIR, ECON_DIR
panel["d_oi"]        = panel["oi"].diff()
panel["oi_zscore"]   = (panel["oi"] - panel["oi"].rolling(720).mean()) / panel["oi"].rolling(720).std()
panel["oi_high"]     = (panel["oi"].rolling(720).rank(pct=True) > (CFG.ECON.high_oi_pctile / 100)).astype(int)

# 3c. OI-to-volume ratio (leverage crowding proxy)
vol_ma = panel["volume_perp"].rolling(24).mean()
panel["oi_vol_ratio"] = panel["oi"] / vol_ma.replace(0, np.nan)

# 3d. Funding rate: already on hourly grid (ffill from settlements)

# 3e. Rolling volatility (7-day, annualized is optional — raw for now)
panel["vol_eth_7d"]   = panel["ret_eth_perp"].rolling(CFG.ECON.vol_window).std()
panel["vol_btc_7d"]   = panel["ret_btc_spot"].rolling(CFG.ECON.vol_window).std()

# 3f. Volatility-normalized returns (for cross-asset comparison)
panel["ret_eth_std"]  = panel["ret_eth_perp"] / panel["vol_eth_7d"].replace(0, np.nan)
panel["ret_btc_std"]  = panel["ret_btc_spot"] / panel["vol_btc_7d"].replace(0, np.nan)

# 3g. Basis: perp vs spot spread
panel["basis_bps"]    = 1e4 * (panel["close_perp"] - panel["close_eth_spot"]) / panel["close_eth_spot"]

print("Transformed variables created:")
new_cols = [c for c in panel.columns if c not in cal.columns and c not in
            ["close_perp", "volume_perp", "oi", "funding_rate", "close_btc_spot", "close_eth_spot"]]
for c in new_cols:
    print(f"  {c}")

Transformed variables created:
  ret_eth_perp
  ret_btc_spot
  ret_eth_spot
  d_oi
  oi_zscore
  oi_high
  oi_vol_ratio
  vol_eth_7d
  vol_btc_7d
  ret_eth_std
  ret_btc_std
  basis_bps


In [15]:
# ── 4. Missing data audit ──

print("\n═══ Missing Data Audit ═══")
key_cols = ["close_perp", "oi", "funding_rate", "close_btc_spot", "close_eth_spot"]
for c in key_cols:
    n = panel[c].isna().sum()
    pct = 100 * n / len(panel)
    print(f"  {c:20s}: {n:5,} missing ({pct:.2f}%)")

# Core window filter
with open(CFG.FILES.window_metadata) as f:
    meta = json.load(f)
core_start = pd.Timestamp(meta["core_window"]["start"])
core_end   = pd.Timestamp(meta["core_window"]["end_excl"])

core = panel[(panel["date"] >= core_start) & (panel["date"] < core_end)].copy()
print(f"\nCore window: {len(core):,} rows  [{core_start}, {core_end})")
print("Core missing:")
for c in key_cols:
    n = core[c].isna().sum()
    print(f"  {c:20s}: {n:5,}")


═══ Missing Data Audit ═══
  close_perp          :     0 missing (0.00%)
  oi                  :     0 missing (0.00%)
  funding_rate        :     0 missing (0.00%)
  close_btc_spot      :     0 missing (0.00%)
  close_eth_spot      :     0 missing (0.00%)

Core window: 41,328 rows  [2021-03-15 00:00:00+00:00, 2025-12-01 00:00:00+00:00)
Core missing:
  close_perp          :     0
  oi                  :     0
  funding_rate        :     0
  close_btc_spot      :     0
  close_eth_spot      :     0


In [17]:
# ── 5. Placeholder columns for DeFi (filled later in nb04) ──

panel["liq_usd_total"]    = np.nan
panel["log_liq"]          = np.nan   # ln(1 + L_t), computed in nb04

print("DeFi placeholder columns added (NaN — will be filled in notebook 04)")

DeFi placeholder columns added (NaN — will be filled in notebook 04)


In [19]:
# ── 6. Save ──

panel.to_parquet(CFG.FILES.econ_core_predefi, index=False, engine="pyarrow")
print(f"\nSaved: {CFG.FILES.econ_core_predefi}")
print(f"Shape: {panel.shape}")

# QA
qa = {
    "n_rows": len(panel),
    "n_cols": panel.shape[1],
    "columns": list(panel.columns),
    "core_window_n": len(core),
    "missing": {c: int(panel[c].isna().sum()) for c in key_cols},
    "status": "PASS (pre-DeFi)",
}
from config import CFG, REPORTS_DIR, ECON_DIR
qa_path = REPORTS_DIR / "econ_core_predefi_qa.json"
with open(qa_path, "w") as f:
    json.dump(qa, f, indent=2)
print(f"Saved: {qa_path}")

print("\n✅ Notebook 03 complete")


Saved: ./data/econ/econ_core_predefi_1h.parquet
Shape: (41328, 21)
Saved: ./data/analysis/reports/econ_core_predefi_qa.json

✅ Notebook 03 complete
